# Encoder Analysis

1. Checking collapse
2. Confounder probe

In [1]:
import numpy as np
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

from src.config import DiffusionConfig, ModelConfig
from src.data import load_ihdp, make_ihdp_confounded
from src.model import DiffPOCEVAE

/home/justin/msc_ai/individual-project/diffusion-irregular-ehr/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_CFG = ModelConfig(feature_dim=25, latent_dim=20, hidden_dim=64, num_layers=2)
DIFF_CFG = DiffusionConfig(
    num_steps=100,
    beta_start=0.0001,
    beta_end=0.2,
    schedule="quad",
    embedding_dim=32,
    block_dim=32,
    hidden_dim=32,
    num_blocks=4,
    clip_denoised=True,
)
CKPT_PATH = "checkpoints/best_model_hybrid_conf_rep1_2026-08-07T16_55_34.pth"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [3]:
train_ds, val_ds, test_ds, y_std = load_ihdp(
    "data/ihdp", replication=1, train_ratio=0.7, test_ratio=0.15
)
train_ds, val_ds, test_ds = (make_ihdp_confounded(ds) for ds in (train_ds, val_ds, test_ds))

model = DiffPOCEVAE(MODEL_CFG, DIFF_CFG).to(device)
model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
model.eval()


def get_mu_and_sigma(ds) -> tuple[torch.Tensor, torch.Tensor]:
    x, a, y = ds.x.to(device), ds.a.to(device), ds.y.to(device)
    with torch.no_grad():
        _, mu, sigma = model.encoder.rsample(x, a, y)
    return mu.cpu(), sigma.cpu()

## Checking Collapse

This section is for checking DiffPOCEVAE's latent `z` for posterior collapse on a `hybrid_*` checkpoint.

If the encoder has collapsed to the prior N(0,I), `mu` will sit near 0 and `sigma` near 1 for most latent dimensions, with little variation across subjects -- meaning `z` carries almost no information about `(x, a, y_fac)`, regardless of `latent_dim`/`hidden_dim` size.

In [4]:
train_ds, val_ds, test_ds, y_std = load_ihdp(
    "data/ihdp", replication=1, train_ratio=0.7, test_ratio=0.15
)
train_ds, val_ds, test_ds = (
    make_ihdp_confounded(ds, effect=0.4) for ds in (train_ds, val_ds, test_ds)
)

model = DiffPOCEVAE(MODEL_CFG, DIFF_CFG).to(device)
model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
model.eval()

mu, sigma = get_mu_and_sigma(train_ds)
print(f"z shape: {mu.shape}  (N={mu.shape[0]}, latent_dim={mu.shape[1]})")

z shape: torch.Size([689, 20])  (N=689, latent_dim=20)


In [5]:
print("Prior is N(0, 1) per dimension. Collapse looks like mu~0, sigma~1, low mu variance.\n")

print(f"{'dim':>4} {'mean|mu|':>10} {'std(mu)':>10} {'mean sigma':>11}")
for d in range(mu.shape[1]):
    print(
        f"{d:>4}"
        f" {mu[:, d].abs().mean().item():>10.4f}"
        f" {mu[:, d].std().item():>10.4f}"
        f" {sigma[:, d].mean().item():>11.4f}"
    )

print()
print(
    f"Aggregate: mean|mu|={mu.abs().mean().item():.4f}  "
    f"mean std(mu) across dims={mu.std(dim=0).mean().item():.4f}  "
    f"mean sigma={sigma.mean().item():.4f}"
)
print(
    f"KL from prior (mean over dims and subjects): "
    f"{(0.5 * (mu.pow(2) + sigma.pow(2) - 2 * sigma.log() - 1)).mean().item():.4f}"
)

Prior is N(0, 1) per dimension. Collapse looks like mu~0, sigma~1, low mu variance.

 dim   mean|mu|    std(mu)  mean sigma
   0     0.4647     0.5306      0.7707
   1     0.6239     0.7605      0.6917
   2     0.4097     0.4773      0.8277
   3     0.4620     0.4938      0.8041
   4     0.4455     0.5487      0.8137
   5     0.3315     0.4144      0.8696
   6     0.2618     0.3209      0.8754
   7     0.3378     0.3736      0.8666
   8     0.4286     0.5367      0.7902
   9     0.4055     0.4797      0.8323
  10     0.6037     0.7063      0.7539
  11     0.4372     0.5191      0.8210
  12     0.7293     0.8507      0.6205
  13     0.4408     0.5410      0.7944
  14     0.4100     0.4913      0.8198
  15     0.4536     0.5166      0.8051
  16     0.3410     0.4124      0.8754
  17     0.4319     0.4890      0.8095
  18     0.6739     0.7917      0.7134
  19     0.4045     0.4927      0.8220

Aggregate: mean|mu|=0.4548  mean std(mu) across dims=0.5374  mean sigma=0.7988
KL from prior (m

## Confounder Probe

This section probes *'does z actually encode the hidden confounder?'*

Trains a logistic regression on `z` (the trained encoder's posterior mean) to predict `momblack`, and compares against the same probe trained directly on `x` -- the raw covariates `z` was derived from. If `x` predicts `momblack` better than `z` does, the encoder is losing confounder-relevant signal during encoding, not just failing to have any (posterior collapse, which has been ruled out above).

In [6]:
x_train, x_test = train_ds.x.numpy(), test_ds.x.numpy()
z_train, _ = get_mu_and_sigma(train_ds)
z_test, _ = get_mu_and_sigma(test_ds)
conf_train = train_ds.confounder.astype(int)
conf_test = test_ds.confounder.astype(int)

In [7]:
base_rate = conf_test.mean()
majority_acc = max(base_rate, 1 - base_rate)
print(f"N train={len(conf_train)}  N test={len(conf_test)}  test base rate={base_rate:.4f}")
print(f"Majority-class baseline accuracy: {majority_acc:.4f}\n")

N train=689  N test=148  test base rate=0.5068
Majority-class baseline accuracy: 0.5068



In [8]:
print(f"{'probe input':<12} {'accuracy':>10} {'AUC':>10}")
for name, xtr, xte in (
    ("x (25-dim)", x_train, x_test),
    ("z (20-dim)", z_train.numpy(), z_test.numpy()),
):
    clf = LogisticRegression(max_iter=2000).fit(xtr, conf_train)
    pred = clf.predict(xte)
    proba = clf.predict_proba(xte)[:, 1]
    acc = accuracy_score(conf_test, pred)
    auc = roc_auc_score(conf_test, proba)
    print(f"{name:<12} {acc:>10.4f} {auc:>10.4f}")

probe input    accuracy        AUC
x (25-dim)       0.7297     0.8405
z (20-dim)       0.7297     0.8203
